# Pareto Frontier / PR Curve: Grid Search for Optimal Prediction Metric Thresholds

对 AF3 置信度指标进行网格搜索，分析 PR 曲线和 F1，同时支持多指标 Pareto 前沿。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CSV = './summary/pro_msa_notemplate-pep_nomsa_notemplate.csv'
df = pd.read_csv(CSV)
df['label'] = df['lRMSD'] < 2.5
print(f'Total: {len(df)}, Label True: {df["label"].sum()}, False: {(~df["label"]).sum()}')
print(f'Unique targets: {df["native"].nunique()}')

In [ ]:
def single_metric_pr(df, metric, thresholds=None, higher_is_better=True):
    """单一指标的PR曲线，计算每个threshold下的precision/recall/F1"""
    if thresholds is None:
        lo, hi = df[metric].quantile([0.05, 0.95])
        thresholds = np.linspace(lo, hi, 20)
    
    results = []
    total_pos = df['label'].sum()
    for t in thresholds:
        mask = df[metric] > t if higher_is_better else df[metric] < t
        kept = df[mask]
        if len(kept) == 0:
            continue
        tp = kept['label'].sum()
        precision = tp / len(kept)
        recall = tp / total_pos
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        results.append({'threshold': t, 'precision': precision, 'recall': recall, 'f1': f1,
                        'kept': len(kept), 'tp': tp})
    
    return pd.DataFrame(results)


metrics = ["pep_plDDT", "plDDT", "ipTM", "pTM", "pep_pTM", "ranking_score", "pAE_min", "ipAE", "ipSAE_max", "ipSAE_min"]
fig, axes = plt.subplots(5, 2, figsize=(15, 30))
axes = axes.flatten()

best_rows = []
for idx, m in enumerate(metrics):
    ax = axes[idx]
    higher = m not in ('pAE_min', 'ipAE')
    res = single_metric_pr(df, m, higher_is_better=higher)
    if len(res) == 0:
        ax.set_title(f'{m} (no data)')
        continue
    
    best = res.loc[res['f1'].idxmax()]
    best_rows.append({'metric': m, 'threshold': best['threshold'],
                      'precision': best['precision'], 'recall': best['recall'],
                      'f1': best['f1'], 'kept': best['kept'], 'tp': best['tp']})
    
    ax.plot(res['recall'], res['precision'], 'b-', linewidth=1.5, alpha=0.7, label='PR curve')
    ax.scatter(best['recall'], best['precision'], c='red', s=50, zorder=5)
    ax.annotate(f'thr={best["threshold"]:.2f}\nF1={best["f1"]:.3f}\nP={best["precision"]:.3f}\nR={best["recall"]:.3f}',
                (best['recall'], best['precision']), fontsize=8, ha='left', va='bottom',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title(f'{m}  {"↑" if higher else "↓"}')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)

plt.tight_layout()
plt.show()

# 最佳F1汇总表
best_df = pd.DataFrame(best_rows).sort_values('f1', ascending=False)
print('\n=== 各指标最佳F1汇总（按F1降序） ===')
print(best_df.to_string(index=False))

In [ ]:
from itertools import product

def multi_metric_grid(df, metrics, n_thresh=8):
    """多指标网格搜索 + Pareto前沿"""
    higher = [m != 'pae_min' for m in metrics]
    thresholds = []
    for m, hi in zip(metrics, higher):
        lo, hi_val = df[m].quantile([0.05, 0.95])
        thresholds.append(np.linspace(lo, hi_val, n_thresh))
    
    total_pos = df['label'].sum()
    results = []
    
    for combo in product(*thresholds):
        mask = pd.Series(True, index=df.index)
        for m, t, hi in zip(metrics, combo, higher):
            mask &= (df[m] > t) if hi else (df[m] < t)
        kept = df[mask]
        if len(kept) == 0:
            continue
        tp = kept['label'].sum()
        precision = tp / len(kept)
        recall = tp / total_pos
        results.append({**dict(zip(metrics, combo)),
                        'precision': precision, 'recall': recall,
                        'kept': len(kept), 'tp': tp})
    
    res = pd.DataFrame(results)
    if len(res) == 0:
        return res, pd.DataFrame()
    
    is_pareto = np.ones(len(res), dtype=bool)
    for i in range(len(res)):
        for j in range(len(res)):
            if i != j:
                r_i, p_i = res.iloc[i]['recall'], res.iloc[i]['precision']
                r_j, p_j = res.iloc[j]['recall'], res.iloc[j]['precision']
                if r_j >= r_i and p_j >= p_i and (r_j > r_i or p_j > p_i):
                    is_pareto[i] = False
                    break
    frontier = res[is_pareto].sort_values('recall')
    return res, frontier

# --- Dual-metric combos ---
metric_cols = ['ipTM', 'pTM', 'pep_plDDT', 'ranking_score', 'pAE_min']
dual_combos = [
    (metric_cols[i], metric_cols[j])
    for i in range(len(metric_cols))
    for j in range(i + 1, len(metric_cols))
]

fig, axes = plt.subplots(5, 2, figsize=(16, 30))
axes = axes.flatten()

for idx, combo in enumerate(dual_combos):
    ax = axes[idx]
    res, frontier = multi_metric_grid(df, list(combo))
    if len(res) == 0:
        ax.set_title(f'{combo[0]}+{combo[1]} (no data)')
        continue
    ax.scatter(res['recall'], res['precision'], s=8, alpha=0.4, label='grid')
    ax.plot(frontier['recall'], frontier['precision'], 'r-', linewidth=2, label='pareto')
    for i, (_, row) in enumerate(frontier.iterrows()):
        if i % 2 == 0:
            label = '+'.join(f'{row[c]:.2f}' for c in combo)
            ax.annotate(label, (row['recall'], row['precision']),
                        fontsize=6, ha='center', va='bottom')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title(f'{combo[0]} + {combo[1]}')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- Triple-metric combos ---
triple_combos = [
    (metric_cols[i], metric_cols[j], metric_cols[k])
    for i in range(len(metric_cols))
    for j in range(i + 1, len(metric_cols))
    for k in range(j + 1, len(metric_cols))
]

fig, axes = plt.subplots(5, 2, figsize=(16, 30))
axes = axes.flatten()
for idx, combo in enumerate(triple_combos):
    ax = axes[idx]
    res, frontier = multi_metric_grid(df, list(combo))
    if len(res) == 0:
        ax.set_title(f'{combo[0]}+{combo[1]}+{combo[2]} (no data)')
        continue
    ax.scatter(res['recall'], res['precision'], s=8, alpha=0.3, label='grid')
    ax.plot(frontier['recall'], frontier['precision'], 'r-', linewidth=2, label='pareto')
    for i, (_, row) in enumerate(frontier.iterrows()):
        if i % 2 == 0:
            label = '+'.join(f'{row[c]:.2f}' for c in combo)
            ax.annotate(label, (row['recall'], row['precision']),
                        fontsize=6, ha='center', va='bottom')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title(f'{combo[0]} + {combo[1]} + {combo[2]}')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
def best_combinations(df, combos, label=''):
    """输出Pareto前沿上按precision/recall/F1排序的最佳组合"""
    all_frontier = []
    for combo in combos:
        _, frontier = multi_metric_grid(df, list(combo))
        if len(frontier) > 0:
            f = frontier.copy()
            f['combo_name'] = '+'.join(combo)
            f['f1'] = 2 * f['precision'] * f['recall'] / (f['precision'] + f['recall'] + 1e-10)
            all_frontier.append(f)
    
    if not all_frontier:
        return
    all_f = pd.concat(all_frontier, ignore_index=True)
    
    print(f'=== {label} ===')
    meta_cols = [c for c in all_f.columns if c not in ('precision','recall','f1','kept','tp','combo_name')]
    display_cols = ['combo_name'] + meta_cols + ['precision','recall','f1','kept']
    display_cols = [c for c in display_cols if c in all_f.columns]
    
    print('\n[Conservative] Top 3 by Precision:')
    print(all_f.nlargest(3, 'precision')[display_cols].to_string(index=False))
    print('\n[Aggressive] Top 3 by Recall:')
    print(all_f.nlargest(3, 'recall')[display_cols].to_string(index=False))
    print('\n[Balanced] Top 3 by F1:')
    print(all_f.nlargest(3, 'f1')[display_cols].to_string(index=False))
    print()

best_combinations(df, dual_combos, 'Dual-metric Pareto frontier')
best_combinations(df, triple_combos, 'Triple-metric Pareto frontier')